#### Problem 10

a. Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of the mean. Report the seed and number of bootstrap replications.

In [4]:

import numpy as np
import pandas as pd

df = pd.read_csv("../data/nhanes_data_17_18.csv", low_memory=False)

adult_data = df[
    (df["AgeInYearsAtScreening"] >= 18) &
    (df["SleepHoursWeekdaysOrWorkdays"].notna())
]

sleep_hours = adult_data["SleepHoursWeekdaysOrWorkdays"].to_numpy()

original_mean = np.mean(sleep_hours)


In [6]:
seed = 100
n_bootstrap = 2000
rng = np.random.default_rng(seed)

bootstrap_means = []

for _ in range(n_bootstrap):
    bootstrap_sample = rng.choice(
        sleep_hours,
        size=len(sleep_hours),
        replace=True
    )

    bootstrap_means.append(np.mean(bootstrap_sample))

bootstrap_means = np.array(bootstrap_means)

print(f"Original mean: {original_mean:.4f} hours")
print(f"Bootstrap mean: {np.mean(bootstrap_means):.4f} hours")
print(f"Seed: {seed}")
print(f"Bootstrap replications: {n_bootstrap}")

Original mean: 7.6212 hours
Bootstrap mean: 7.6214 hours
Seed: 100
Bootstrap replications: 2000


b. Compute the bootstrap standard error as the sample standard deviation of the bootstrap estimates (ddof=1).

In [8]:
bootstrap_se = np.std(bootstrap_means, ddof=1)

print(f"Bootstrap SE: {bootstrap_se:.4f} hours")


Bootstrap SE: 0.0222 hours


c. Using the original estimate and the bootstrap standard error, construct centered-Normal 90%, 95%, and 99% confidence intervals.

In [10]:
critical_values = {
    0.90: 1.645,
    0.95: 1.96,
    0.99: 2.576
}

confidence_intervals = {}

for confidence, critical_value in critical_values.items():
    margin_error = critical_value * bootstrap_se

    lower = original_mean - margin_error
    upper = original_mean + margin_error

    confidence_intervals[confidence] = (lower, upper)

    print(f"{confidence:.0%} CI: [{lower:.4f}, {upper:.4f}]")


90% CI: [7.5847, 7.6577]
95% CI: [7.5778, 7.6647]
99% CI: [7.5641, 7.6784]


d. Use each confidence interval to evaluate $H_0: M = 8$. At the matching significance levels $\alpha = 0.10, 0.05, 0.01$, state whether you reject or fail to reject the null hypothesis.

In [11]:
null_mean = 8

for confidence, (lower, upper) in confidence_intervals.items():
    alpha = 1 - confidence

    if lower <= null_mean <= upper:
        decision = "Fail to reject H0"
    else:
        decision = "Reject H0"

    print(f"Alpha = {alpha:.2f}: {decision}")


Alpha = 0.10: Reject H0
Alpha = 0.05: Reject H0
Alpha = 0.01: Reject H0


e. Interpret the result and discuss whether its statistical and practical significance tell the same story. For this exercise, treat observations as independent and unweighted. Because the complex survey design is ignored, do not describe the result as nationally representative or the intervals as survey-adjusted.

According to our results, for our dataset, we have enough evidence to say there is a statistically significant difference from 8 hours. However, in a practical setting, the difference is estimated to be around half an hour, which may still be considered close to 8 hours.